# AlexNet Architecture - Layer-by-Layer Explanation

AlexNet is a deep convolutional neural network architecture that won the ImageNet 2012 challenge. It takes an RGB image as input and classifies it into 1000 categories.

---
 
## 🔹 Input
- **Size:** `227 × 227 × 3` (RGB image)

---

## 🔹 1st Convolutional Layer
- **Filter Size:** `11 × 11`
- **Stride:** `4`
- **Padding:** None
- **Number of Filters:** `96`
- **Output Size Calculation:**

  ```
  Output = ⌊ (227 - 11) / 4 + 1 ⌋
         = ⌊ 216 / 4 + 1 ⌋
         = ⌊ 54 + 1 ⌋ = 55
  ```

- ✅ **Output Size:** `55 × 55 × 96`

---

## 🔹 1st Max Pooling Layer
- **Filter Size:** `3 × 3`
- **Stride:** `2`
- **Output Size Calculation:**

  ```
  Output = ⌊ (55 - 3) / 2 + 1 ⌋
         = ⌊ 52 / 2 + 1 ⌋
         = ⌊ 26 + 1 ⌋ = 27
  ```

- ✅ **Output Size:** `27 × 27 × 96`

---

## 🔹 2nd Convolutional Layer
- **Filter Size:** `5 × 5`
- **Padding:** SAME (padding = 2)
- **Number of Filters:** `256`
- ✅ **Output Size:** `27 × 27 × 256`

---

## 🔹 2nd Max Pooling Layer
- **Filter Size:** `3 × 3`
- **Stride:** `2`
- **Output Size Calculation:**

  ```
  Output = ⌊ (27 - 3) / 2 + 1 ⌋
         = ⌊ 24 / 2 + 1 ⌋
         = ⌊ 12 + 1 ⌋ = 13
  ```

- ✅ **Output Size:** `13 × 13 × 256`

---

## 🔹 3rd Convolutional Layer
- **Filter Size:** `3 × 3`
- **Padding:** SAME
- **Number of Filters:** `384`
- ✅ **Output Size:** `13 × 13 × 384`

---

## 🔹 4th Convolutional Layer
- **Filter Size:** `3 × 3`
- **Padding:** SAME
- **Number of Filters:** `384`
- ✅ **Output Size:** `13 × 13 × 384`

---

## 🔹 5th Convolutional Layer
- **Filter Size:** `3 × 3`
- **Padding:** SAME
- **Number of Filters:** `256`
- ✅ **Output Size:** `13 × 13 × 256`

---

## 🔹 3rd Max Pooling Layer
- **Filter Size:** `3 × 3`
- **Stride:** `2`
- **Output Size Calculation:**

  ```
  Output = ⌊ (13 - 3) / 2 + 1 ⌋
         = ⌊ 10 / 2 + 1 ⌋
         = ⌊ 5 + 1 ⌋ = 6
  ```

- ✅ **Output Size:** `6 × 6 × 256`

---

## 🔹 Flatten Layer
- Flatten: `6 × 6 × 256 = 9216`
- ✅ Output: `Vector of size 9216`

---

## 🔹 Fully Connected (Dense) Layers

| Layer             | Input Units | Output Units |
|------------------|-------------|--------------|
| Fully Connected 1| 9216        | 4096         |
| Fully Connected 2| 4096        | 4096         |
| Fully Connected 3| 4096        | 1000 (Softmax)|

---

## 🔹 Summary Table of Output Sizes

| Layer                  | Output Size         |
|------------------------|---------------------|
| Input                  | 227 × 227 × 3       |
| Conv1 (11×11, s=4)     | 55 × 55 × 96        |
| MaxPool1 (3×3, s=2)    | 27 × 27 × 96        |
| Conv2 (5×5, SAME)      | 27 × 27 × 256       |
| MaxPool2 (3×3, s=2)    | 13 × 13 × 256       |
| Conv3 (3×3, SAME)      | 13 × 13 × 384       |
| Conv4 (3×3, SAME)      | 13 × 13 × 384       |
| Conv5 (3×3, SAME)      | 13 × 13 × 256       |
| MaxPool3 (3×3, s=2)    | 6 × 6 × 256         |
| Flatten                | 9216                |
| FC1                    | 4096                |
| FC2                    | 4096                |
| FC3 (Output)           | 1000                |

---

## 🔸 Total Parameters
- **Approximate Parameters:** ~60 million


In [1]:
# ✅ Step 1: Unzip archive.zip (uploaded to Colab)
import zipfile
import os

with zipfile.ZipFile("archive.zip", "r") as zip_ref:
    zip_ref.extractall()

In [2]:
# ✅ Step 2: Imports
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

In [3]:
# ✅ Step 3: Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ✅ Step 4: Data transforms
transform = transforms.Compose([
    transforms.Resize((227, 227)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

Device: cuda


In [4]:
# ✅ Step 5: Load dataset
data_dir = "PetImages"
dataset = datasets.ImageFolder(data_dir, transform=transform)

In [ ]:
# ✅ Step 6: Remove corrupted images
valid_samples = []
for path, label in dataset.samples:
    try:
        img = Image.open(path)
        img.verify()
        valid_samples.append((path, label))
    except:
        continue

dataset.samples = valid_samples
print(f"✅ Dataset size after filtering: {len(dataset)}")

In [5]:
# ✅ Step 7: Dataset split
train_size = int(0.7 * len(dataset))
val_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - val_size
train_ds, val_ds, test_ds = random_split(dataset, [train_size, val_size, test_size])

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=64, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=64, num_workers=0)


In [ ]:
# ✅ Step 8: AlexNet model
class AlexNet(nn.Module):
    def __init__(self, num_classes=2):
        super(AlexNet, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 96, kernel_size=11, stride=4),
            nn.ReLU(inplace=True), #
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Conv2d(96, 256, kernel_size=5, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Conv2d(256, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(384, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2)
        )
        self.classifier = nn.Sequential(
            nn.Dropout(),
            nn.Linear(256 * 6 * 6, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Linear(4096, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

In [ ]:
# ✅ Step 9: Model, loss, optimizer
model = AlexNet(num_classes=2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

# ✅ Step 10: Training function
def train(model, loader):
    model.train() # Set model to training mode 
    total_loss = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        pred = model(x)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


In [8]:
# ✅ Step 11: Evaluation function
def evaluate(model, loader):
    model.eval()
    correct = 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            _, preds = torch.max(out, 1)
            correct += (preds == y).sum().item()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
    return correct / len(loader.dataset), all_preds, all_labels


In [9]:
# ✅ Step 12: Train loop
for epoch in range(10):
    loss = train(model, train_loader)
    acc, _, _ = evaluate(model, val_loader)
    print(f"Epoch {epoch+1}: Train Loss = {loss:.4f}, Val Acc = {acc:.4f}")

/usr/local/lib/python3.11/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Epoch 1: Train Loss = 0.6484, Val Acc = 0.7157
Epoch 2: Train Loss = 0.5282, Val Acc = 0.7927
Epoch 3: Train Loss = 0.4190, Val Acc = 0.8370
Epoch 4: Train Loss = 0.3495, Val Acc = 0.8538
Epoch 5: Train Loss = 0.2951, Val Acc = 0.8760
Epoch 6: Train Loss = 0.2448, Val Acc = 0.8818
Epoch 7: Train Loss = 0.2006, Val Acc = 0.8906
Epoch 8: Train Loss = 0.1678, Val Acc = 0.8882
Epoch 9: Train Loss = 0.1289, Val Acc = 0.8960
Epoch 10: Train Loss = 0.1060, Val Acc = 0.8914


In [10]:
# ✅ Step 13: Final test accuracy
test_acc, preds, labels = evaluate(model, test_loader)
print(f"\nTest Accuracy = {test_acc:.4f}")
print("\nClassification Report:\n")
print(classification_report(labels, preds, target_names=dataset.classes))


Test Accuracy = 0.8923

Classification Report:

              precision    recall  f1-score   support

         Cat       0.92      0.86      0.89      1864
         Dog       0.87      0.93      0.90      1887

    accuracy                           0.89      3751
   macro avg       0.89      0.89      0.89      3751
weighted avg       0.89      0.89      0.89      3751

